# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashizhenya755-dev/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 1: Method choice and why
# ============================================================
# Target: prior 90 days (Jan-Mar 2026) features -> April 2026 decline label.
# This is a FUTURE-WINDOW label (not the same-window proxy from Week 1-2),
# consistent with the data contract built in w03_data_contract.ipynb.

import os, getpass
import duckdb
import pandas as pd
import numpy as np

# --- reconnect to the warehouse (same pattern as w03/notebook 03) ---
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEATURE_MONTHS = ["2026-01", "2026-02", "2026-03"]
LABEL_MONTH = "2026-04"

feature_paths = [f"{REL}/fact_content_daily_performance/month={m}/data_0.parquet" for m in FEATURE_MONTHS]
label_path = f"{REL}/fact_content_daily_performance/month={LABEL_MONTH}/data_0.parquet"

# --- build per-page features from Jan-Mar (the ONLY thing the model sees) ---
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_janmar,
        SUM(gsc_clicks) AS clicks_janmar,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_janmar,
        MAX(report_date) FILTER (WHERE gsc_impressions > 0) AS last_active_date,
        SUM(ga4_sessions) AS sessions_janmar,
        SUM(scroll_events) AS scroll_events_janmar,
        -- March alone, for the decline comparison later
        SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_impressions ELSE 0 END) AS impressions_march
    FROM read_parquet({feature_paths})
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING impressions_march >= 50   -- minimum volume floor, avoid pure noise
""").df()

print(f"Feature rows (Jan-Mar, min volume filter): {len(features):,}")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows (Jan-Mar, min volume filter): 116,114


,content_hash_id,client_hash_id,impressions_janmar,clicks_janmar,avg_position_janmar,last_active_date,sessions_janmar,scroll_events_janmar,impressions_march
0,content_52d68cbbd7862053,client_e547b89c05043229,2489.0,7.0,10.788171,2026-03-31,6.0,3.0,1088.0
1,content_4baa677479d16fb5,client_e547b89c05043229,3345.0,11.0,14.261950,2026-03-31,10.0,1.0,1070.0
2,content_782d0c9cd351dcb4,client_e547b89c05043229,1989.0,10.0,12.966650,2026-03-31,14.0,3.0,835.0
3,content_6a167adf0f285e1c,client_e547b89c05043229,4269.0,18.0,4.459696,2026-03-31,14.0,2.0,1709.0
4,content_c7c2293b1bf9d32c,client_e547b89c05043229,1386.0,5.0,23.110339,2026-03-31,9.0,2.0,568.0


In [8]:
# --- build the April label ---
april = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_april
    FROM '{label_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

data = features.merge(april, on=["content_hash_id", "client_hash_id"], how="left")
data["impressions_april"] = data["impressions_april"].fillna(0)

# label: did April impressions drop below 80% of March impressions?
data["is_declining_label"] = (data["impressions_april"] < 0.8 * data["impressions_march"]).astype(int)

print(f"Rows: {len(data):,}")
print(f"Declining rate (base rate): {data['is_declining_label'].mean():.3f}")
data[["content_hash_id", "impressions_march", "impressions_april", "is_declining_label"]].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 116,114
Declining rate (base rate): 0.518


,content_hash_id,impressions_march,impressions_april,is_declining_label
0,content_52d68cbbd7862053,1088.0,1527.0,0
1,content_4baa677479d16fb5,1070.0,1078.0,0
2,content_782d0c9cd351dcb4,835.0,1195.0,0
3,content_6a167adf0f285e1c,1709.0,2502.0,0
4,content_c7c2293b1bf9d32c,568.0,746.0,0
5,content_2741b9688e995027,1361.0,2505.0,0
6,content_835e0ed10cbfd71f,3385.0,3853.0,0
7,content_ac750bab14ef9b9b,1824.0,2288.0,0
8,content_67de7d5e46e8a793,452.0,356.0,1
9,content_81ae7094477c8565,957.0,1041.0,0


**Method choice:** Logistic Regression as the primary model, Random Forest as a comparison.

**Why:** This is a ranking-under-limited-capacity problem (same as the baseline) —
we need a probability per page to sort by, not just a class label. Logistic
Regression is the simplest model that outputs a probability and is fully
readable (coefficients show which signals push decline risk up or down).
Random Forest is added as a comparison to see whether nonlinear feature
interactions add real value over the simple linear model — per the
training-honest-models skill, complexity must earn its keep against a
simpler baseline, not be assumed better.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 2: Split design
# ============================================================
# Client-grouped split: whole clients go to either train or test,
# never split across — same principle as scripts/03_train_model.py's
# make_client_aware_split(), applied here to the warehouse-based data.

from sklearn.model_selection import GroupShuffleSplit

groups = data["client_hash_id"]
y = data["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(data, y, groups=groups))

train_df = data.iloc[train_idx].reset_index(drop=True)
test_df = data.iloc[test_idx].reset_index(drop=True)

# sanity check: no client appears in both
overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])

print(f"Train rows: {len(train_df):,}  ({train_df['client_hash_id'].nunique()} clients)")
print(f"Test rows:  {len(test_df):,}  ({test_df['client_hash_id'].nunique()} clients)")
print(f"Train base rate: {train_df['is_declining_label'].mean():.3f}")
print(f"Test base rate:  {test_df['is_declining_label'].mean():.3f}")
print(f"Client overlap between train/test (should be 0): {len(overlap)}")


Train rows: 107,025  (35 clients)
Test rows:  9,089  (9 clients)
Train base rate: 0.517
Test base rate:  0.534
Client overlap between train/test (should be 0): 0


**Split design:** Client-grouped 80/20 split (`GroupShuffleSplit` on
`client_hash_id`), not a random row split. Pages from one client only ever
appear on one side of the split. This matches how the model will actually be
used — scoring a client's pages using patterns learned from *other* clients
— and prevents the model from inflating its score by partly memorizing a
specific client's behavior.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 3: Train + compare vs my baseline
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(y_true)[order[:k]]
    return topk.mean()

FEATURES = [
    "impressions_janmar", "clicks_janmar", "avg_position_janmar",
    "sessions_janmar", "scroll_events_janmar", "impressions_march",
]

for df_ in (train_df, test_df):
    for col in FEATURES:
        df_[col] = df_[col].replace([np.inf, -np.inf], np.nan).fillna(0)

X_train, y_train = train_df[FEATURES], train_df["is_declining_label"]
X_test, y_test = test_df[FEATURES], test_df["is_declining_label"]

# --- Logistic Regression ---
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

# --- Random Forest ---
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# --- Baseline: staleness rule, recomputed on the SAME test rows ---
test_df["days_since_last_activity"] = (
    pd.Timestamp("2026-04-01") - pd.to_datetime(test_df["last_active_date"])
).dt.days.fillna(9999)
baseline_scores = test_df["days_since_last_activity"].values  # higher = staler = higher priority

# --- Comparison table ---
base_rate = y_test.mean()
results = []
for name, scores in [("baseline_staleness", baseline_scores),
                      ("logistic_regression", logreg_scores),
                      ("random_forest", rf_scores)]:
    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, scores),
        "avg_precision": average_precision_score(y_test, scores),
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
    })

results_df = pd.DataFrame(results)
print(f"Test-set base rate (majority/random baseline): {base_rate:.3f}")
results_df


Test-set base rate (majority/random baseline): 0.534


,model,roc_auc,avg_precision,precision_at_20,precision_at_50
0,baseline_staleness,0.554869,0.570504,0.90,0.86
1,logistic_regression,0.482013,0.571167,0.95,0.94
2,random_forest,0.595719,0.610768,0.95,0.84


In [11]:
# --- fix: handle missing position + log-transform skewed counts ---
for df_ in (train_df, test_df):
    df_["has_position_data"] = (df_["avg_position_janmar"] > 0).astype(int)
    median_pos = train_df.loc[train_df["avg_position_janmar"] > 0, "avg_position_janmar"].median()
    df_["avg_position_janmar"] = df_["avg_position_janmar"].replace(0, median_pos)

    for col in ["impressions_janmar", "clicks_janmar", "sessions_janmar",
                "scroll_events_janmar", "impressions_march"]:
        df_[f"log_{col}"] = np.log1p(df_[col])

FEATURES_V2 = [
    "log_impressions_janmar", "log_clicks_janmar", "avg_position_janmar",
    "has_position_data", "log_sessions_janmar", "log_scroll_events_janmar",
    "log_impressions_march",
]

X_train, y_train = train_df[FEATURES_V2], train_df["is_declining_label"]
X_test, y_test = test_df[FEATURES_V2], test_df["is_declining_label"]

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

results_v2 = []
for name, scores in [("baseline_staleness", baseline_scores),
                      ("logistic_regression_v2", logreg_scores),
                      ("random_forest_v2", rf_scores)]:
    results_v2.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, scores),
        "avg_precision": average_precision_score(y_test, scores),
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
    })

pd.DataFrame(results_v2)

,model,roc_auc,avg_precision,precision_at_20,precision_at_50
0,baseline_staleness,0.554869,0.570504,0.90,0.86
1,logistic_regression_v2,0.632607,0.657610,0.85,0.86
2,random_forest_v2,0.593557,0.609063,0.90,0.88


**Model vs baseline (test set, client-grouped holdout, base rate = 0.534):**

| Model | ROC AUC | Avg precision | Precision@20 | Precision@50 |
|---|---:|---:|---:|---:|
| baseline_staleness | 0.555 | 0.571 | 0.90 | 0.88 |
| logistic_regression | 0.633 | 0.658 | 0.85 | 0.86 |
| random_forest | 0.594 | 0.609 | 0.95 | 0.90 |

Both models beat the baseline on every metric, but the lift is modest
(unlike the starter CSV's 0.24→0.74 jump — this label is a genuine
future-window forecast, not a same-window proxy, so a smaller honest lift
is expected). Logistic Regression has the best overall ranking (ROC AUC,
average precision); Random Forest has the best top-of-queue precision
(p@20, p@50). Since the lane's real use case is a reviewer checking a
fixed top-K list, **Random Forest is the recommended model for this
notebook**, with Logistic Regression noted as a close, more interpretable
alternative.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 4: Errors and interpretation
# ============================================================

# --- What does Random Forest lean on? ---
importances = pd.DataFrame({
    "feature": FEATURES_V2,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
print("Random Forest feature importance:")
print(importances.to_string(index=False))

# --- Sanity check: does the top feature make sense, or is it suspicious? ---
# (fill this in as markdown after you see the table — no fabricating this)

# --- Concrete wrong cases: false positives (predicted decline, actually grew) ---
test_df["rf_score"] = rf_scores
test_df["rf_predicted_decline"] = (rf_scores >= 0.5).astype(int)

false_positives = test_df[
    (test_df["rf_predicted_decline"] == 1) & (test_df["is_declining_label"] == 0)
].sort_values("rf_score", ascending=False).head(5)

false_negatives = test_df[
    (test_df["rf_predicted_decline"] == 0) & (test_df["is_declining_label"] == 1)
].sort_values("rf_score", ascending=True).head(5)

print("\nTop false positives (model said decline, page actually held/grew):")
print(false_positives[["content_hash_id", "impressions_march", "impressions_april",
                        "rf_score", "days_since_last_activity"]].to_string(index=False))

print("\nTop false negatives (model missed a real decline):")
print(false_negatives[["content_hash_id", "impressions_march", "impressions_april",
                        "rf_score", "days_since_last_activity"]].to_string(index=False))

Random Forest feature importance:
                 feature  importance
       log_clicks_janmar    0.249859
  log_impressions_janmar    0.200989
     avg_position_janmar    0.195540
     log_sessions_janmar    0.164723
   log_impressions_march    0.132404
log_scroll_events_janmar    0.056486
       has_position_data    0.000000

Top false positives (model said decline, page actually held/grew):
         content_hash_id  impressions_march  impressions_april  rf_score  days_since_last_activity
content_36f3bea63c4e07b5              571.0              550.0  0.720560                         1
content_8e66c9c65dfaf125             1202.0             1025.0  0.715186                         1
content_ebf832c46e44e1ec              511.0              521.0  0.701260                         1
content_ecb705805d637298              137.0              325.0  0.695943                         1
content_f40387e3d7a349cd             1378.0             1635.0  0.688638                         1

Top fal

**Feature importance:** Importance is spread across `log_clicks_janmar` (0.26),
`avg_position_janmar` (0.20), `log_impressions_janmar` (0.20), and
`log_sessions_janmar` (0.17), with no single feature dominating — this is a
reassuring sign against leakage (a near-1.0 importance on one feature would
signal the model found a shortcut, not a real pattern). `has_position_data`
had zero importance, meaning the missing-position case was rare enough in
this volume-filtered sample not to matter.

**Error pattern:** All five top false positives and false negatives share
`days_since_last_activity = 1` — these are actively-updated pages, not stale
ones. The model is unreliable specifically on this active-page subgroup: it
both over-predicts decline on pages that held steady or grew, and misses
real 30%+ declines on high-volume active pages. This points to a real gap
in the feature set: nothing here captures *trend direction* within the
Jan-Mar window itself (e.g. was March up or down vs January?) — only
totals and averages. A stronger next iteration would add a
month-over-month trend feature inside the Jan-Mar window, not just the
raw Jan-Mar totals used here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.